In [ ]:
from pathlib import Path
import pickle
from contextlib import contextmanager

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step7_ModelTraining')
    nested_release_dirs = ('Global-solid-waste-prediction',)
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        for release_dir in nested_release_dirs:
            release_candidate = candidate / release_dir
            if all((release_candidate / name).is_dir() for name in sentinel_dirs):
                return release_candidate
    raise FileNotFoundError('Could not locate GitHub release root from the current working directory')

release_root = find_release_root(cwd)
step_root = release_root / 'Step7_ModelTraining'
step2_input_path = release_root / 'Step2_WDI_Features' / '2_Results' / '1_imputed_features_data.csv'
output_dir = step_root / '2_Results'
TARGET_COL = 'Target_Log'
SEED = 42
WASTE_TYPES = ['AW', 'CDW', 'IW', 'MSW']
TARGET_COLS = ['AW_t', 'CDW_t', 'IW_t', 'MSW_t']
FEATURE_CONTRACT = [
    'NY.GDP.MKTP.PP.KD',
    'SP.POP.TOTL',
    'NY.GDP.PCAP.PP.KD',
    'EN.POP.DNST',
    'Proxy_NY.GDP.MKTP.PP.KD_log1p',
    'Proxy_SP.POP.TOTL_log1p',
    'Proxy_NY.GDP.PCAP.PP.KD_log1p',
    'Proxy_EN.POP.DNST_log1p',
    'EN.GHG.CO2.MT.CE.AR5',
    'Proxy_EN.GHG.CO2.MT.CE.AR5_log1p',
]
RAW_CONTEXT_COLUMNS = [
    'Country Code',
    'Country Name',
    'Year',
    'IncomeGroup',
    'Region',
    'WasteFlag',
]
LABEL_RAW_COL = 'Label_Raw'
LABEL_LOG_COL = 'Label_Log'
OBS_FLAG_COL = 'HasObservedLabel'
ROW_ROLE_COL = 'RowRole'
TRAIN_ROLE = 'training'
PRED_ROLE = 'prediction_base'
LOG_PROXY_BASES = [
    'NY.GDP.MKTP.PP.KD',
    'SP.POP.TOTL',
    'NY.GDP.PCAP.PP.KD',
    'EN.POP.DNST',
    'EN.GHG.CO2.MT.CE.AR5',
]
output_dir

In [ ]:
core_features = FEATURE_CONTRACT.copy()
bad_core_features = [col for col in core_features if col.startswith('Is_') or '_x_' in col]
if bad_core_features:
    raise RuntimeError(f'Unexpected routed or flag columns in per-waste contract: {bad_core_features}')
df_wide = pd.read_csv(step2_input_path)
df_wide = df_wide.sort_values(['Country Code', 'Year']).reset_index(drop=True)
len(core_features), df_wide.shape

In [ ]:
base_features = [col for col in df_wide.columns if col not in TARGET_COLS]
chunks = []
for target_col in TARGET_COLS:
    chunk = df_wide[base_features + [target_col]].copy()
    chunk['Waste_Generation_t'] = pd.to_numeric(chunk[target_col], errors='coerce')
    chunk = chunk.drop(columns=[target_col])
    chunk['WasteFlag'] = target_col.replace('_t', '')
    chunks.append(chunk)
df_long = pd.concat(chunks, ignore_index=True).sort_values(['Country Code', 'Year', 'WasteFlag']).reset_index(drop=True)
for col in LOG_PROXY_BASES:
    if col not in df_long.columns:
        continue
    base = pd.to_numeric(df_long[col], errors='coerce').astype(float)
    df_long[col] = base
    proxy_col = f'Proxy_{col}_log1p'
    if proxy_col in core_features:
        df_long[proxy_col] = np.log1p(base.where(base >= 0.0, 0.0))
missing_features = [col for col in core_features if col not in df_long.columns]
if missing_features:
    raise RuntimeError(f'Missing required core features after long-table rebuild: {missing_features}')
df_long[core_features] = df_long[core_features].apply(pd.to_numeric, errors='coerce')
contract_nan_counts = df_long[core_features].isna().sum()
contract_nan_counts = contract_nan_counts[contract_nan_counts > 0]
if not contract_nan_counts.empty:
    raise RuntimeError(
        f'Raw feature contract contains NaN values after numeric coercion: {contract_nan_counts.to_dict()}'
    )
df_long.shape

In [ ]:
import scipy
import sklearn.utils

if not hasattr(scipy, 'interp'):
    scipy.interp = np.interp

@contextmanager
def _print_elapsed_time(*args, **kwargs):
    yield

if not hasattr(sklearn.utils, '_print_elapsed_time'):
    sklearn.utils._print_elapsed_time = _print_elapsed_time

from pycaret.regression import get_config, setup

def fit_pycaret_pipeline(df_train, feature_columns, seed):
    setup(
        data=df_train[feature_columns + [TARGET_COL]].copy(),
        target=TARGET_COL,
        keep_features=feature_columns,
        imputation_type='simple',
        numeric_imputation='median',
        normalize=True,
        remove_multicollinearity=False,
        feature_selection=False,
        fold_strategy='timeseries',
        fold=3,
        data_split_shuffle=False,
        fold_shuffle=False,
        train_size=0.999999,
        memory=False,
        verbose=False,
        html=False,
        session_id=seed,
    )
    return get_config('pipeline')

def transform_with_pipeline(pipeline, frame, input_columns, expected_columns=None):
    transformed = pipeline.transform(frame[input_columns].copy())
    if not isinstance(transformed, pd.DataFrame):
        columns = expected_columns if expected_columns is not None else input_columns
        transformed = pd.DataFrame(transformed, columns=columns)
    transformed = transformed[expected_columns].copy() if expected_columns is not None else transformed.copy()
    if int(transformed.isna().sum().sum()) != 0:
        raise ValueError('Transformed feature matrix still contains NaN')
    return transformed

def build_master_raw_frame(frame, raw_feature_columns, row_role):
    label_raw = pd.to_numeric(frame['Waste_Generation_t'], errors='coerce')
    out = frame[RAW_CONTEXT_COLUMNS + raw_feature_columns].copy()
    out[LABEL_RAW_COL] = label_raw
    out[LABEL_LOG_COL] = np.where(
        label_raw.notna(),
        np.log1p(label_raw.astype(float)),
        np.nan,
    )
    out[OBS_FLAG_COL] = label_raw.notna()
    out[ROW_ROLE_COL] = row_role
    return out

def build_master_processed_frame(raw_master, processed_features, processed_columns):
    left = raw_master[RAW_CONTEXT_COLUMNS + [LABEL_RAW_COL, LABEL_LOG_COL, OBS_FLAG_COL, ROW_ROLE_COL]].reset_index(drop=True)
    right = processed_features[processed_columns].reset_index(drop=True)
    return pd.concat([left, right], axis=1)

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'feature': core_features}).to_csv(output_dir / '0_feature_contract_raw.csv', index=False)
df_wide.to_csv(output_dir / '0_original_wide.csv', index=False)
df_long.to_csv(output_dir / '0_full_long_table.csv', index=False)
processed_contract_written = False
processed_contract = None
summary_rows = []
for waste_type in WASTE_TYPES:
    waste_long = df_long[df_long['WasteFlag'] == waste_type].copy().reset_index(drop=True)

    prediction_raw = build_master_raw_frame(waste_long, core_features, PRED_ROLE)
    training_raw = prediction_raw[prediction_raw[OBS_FLAG_COL]].copy().reset_index(drop=True)
    training_raw[ROW_ROLE_COL] = TRAIN_ROLE

    waste_train = waste_long[waste_long['Waste_Generation_t'].notna()].copy().reset_index(drop=True)
    waste_train[TARGET_COL] = np.log1p(waste_train['Waste_Generation_t'].astype(float))

    pipeline = fit_pycaret_pipeline(waste_train, core_features, SEED)
    with open(output_dir / f'0_preprocessing_pipeline_{waste_type}.pkl', 'wb') as handle:
        pickle.dump(pipeline, handle)

    train_processed_features = transform_with_pipeline(pipeline, waste_train, core_features)
    transformed_columns = train_processed_features.columns.tolist()
    bad_transformed = [col for col in transformed_columns if col.startswith('Is_') or '_x_' in col]
    if bad_transformed:
        raise RuntimeError(f'Unexpected routed or flag columns after preprocessing: {bad_transformed}')

    if not processed_contract_written:
        processed_contract = transformed_columns.copy()
        pd.DataFrame({'feature': processed_contract}).to_csv(output_dir / '0_feature_contract_processed.csv', index=False)
        processed_contract_written = True
    elif transformed_columns != processed_contract:
        raise RuntimeError(f'Processed contract drift detected for {waste_type}: {transformed_columns} != {processed_contract}')

    prediction_processed_features = transform_with_pipeline(pipeline, waste_long, core_features, transformed_columns)

    training_processed = build_master_processed_frame(training_raw, train_processed_features, transformed_columns)
    prediction_processed = build_master_processed_frame(prediction_raw, prediction_processed_features, transformed_columns)

    training_raw.to_csv(output_dir / f'0_training_raw_{waste_type}.csv', index=False)
    training_processed.to_csv(output_dir / f'0_training_processed_{waste_type}.csv', index=False)
    prediction_raw.to_csv(output_dir / f'0_prediction_raw_{waste_type}.csv', index=False)
    prediction_processed.to_csv(output_dir / f'0_prediction_processed_{waste_type}.csv', index=False)

    summary_rows.append({
        'WasteFlag': waste_type,
        'TrainRows': int(len(training_raw)),
        'AllRows': int(len(prediction_raw)),
        'PredictRows': int((~prediction_raw[OBS_FLAG_COL]).sum()),
    })
summary = pd.DataFrame(summary_rows)
summary.to_csv(output_dir / '0_per_waste_data_summary.csv', index=False)
summary